In [6]:
file_schema = """
    id int,
    name string,
    dop string,
    phone long,
    amount string,
    discount string
"""

sales_raw_df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(file_schema).
        load(path = "/workspaces/pyspark_udemy_codespace/data/sales_sample.csv")
)

sales_raw_df.show()
sales_raw_df.printSchema()

+---+--------+----------+----------+----------+--------+
| id|    name|       dop|     phone|    amount|discount|
+---+--------+----------+----------+----------+--------+
|100|Prashant|2020-06-15|9238614990|     12000|    18.5|
|101|   David| 2018-08-7|8908617610|     15000|     nil|
|102|  Simran|14-05-2019|      NULL|3000000000|      21|
+---+--------+----------+----------+----------+--------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dop: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- amount: string (nullable = true)
 |-- discount: string (nullable = true)



In [8]:
sales_raw_df.describe().show()

+-------+-----+------+----------+--------------------+--------------------+------------------+
|summary|   id|  name|       dop|               phone|              amount|          discount|
+-------+-----+------+----------+--------------------+--------------------+------------------+
|  count|    3|     3|         3|                   2|                   3|                 3|
|   mean|101.0|  NULL|      NULL|         9.0736163E9|          1.000009E9|             19.75|
| stddev|  1.0|  NULL|      NULL|2.3334338517179397E8|1.7320430133408928E9|1.7677669529663689|
|    min|  100| David|14-05-2019|          8908617610|               12000|              18.5|
|    max|  102|Simran|2020-06-15|          9238614990|          3000000000|               nil|
+-------+-----+------+----------+--------------------+--------------------+------------------+



In [ ]:
# PROBLEMS:
# Convert id from integer to string and rename it as transaction_id.
# Rename the name column to customer_name.
# Convert the dop to date format and rename the column to date_of_purchase.
# Rename the phone column to customer_phone
# Convert the amount to a long value and filter out nulls and outlier values.
# Rename the column to purchase_amount
# Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

In [17]:
# Transform
from pyspark.sql.functions import expr

sales_df = sales_raw_df.selectExpr(
    "cast(id as string) as transaction_id",
    "name as customer_name",
    "nvl(try_cast(dop as date), to_date(dop, 'dd-MM-yyyy')) as dop",
    "cast(phone as string) as customer_phone",
    "cast(amount as long) as purchase_amount",
    "nvl(try_cast(discount as double), 0) as applied_discount"
).filter("purchase_amount is not null and purchase_amount < 200000")

sales_df.show()

+--------------+-------------+----------+--------------+---------------+----------------+
|transaction_id|customer_name|       dop|customer_phone|purchase_amount|applied_discount|
+--------------+-------------+----------+--------------+---------------+----------------+
|           100|     Prashant|2020-06-15|    9238614990|          12000|            18.5|
|           101|        David|2018-08-07|    8908617610|          15000|             0.0|
+--------------+-------------+----------+--------------+---------------+----------------+



In [18]:
# Verify Statistics

sales_df.describe().show()

+-------+------------------+-------------+--------------------+------------------+-----------------+
|summary|    transaction_id|customer_name|      customer_phone|   purchase_amount| applied_discount|
+-------+------------------+-------------+--------------------+------------------+-----------------+
|  count|                 2|            2|                   2|                 2|                2|
|   mean|             100.5|         NULL|         9.0736163E9|           13500.0|             9.25|
| stddev|0.7071067811865476|         NULL|2.3334338517179397E8|2121.3203435596424|13.08147545195113|
|    min|               100|        David|          8908617610|             12000|              0.0|
|    max|               101|     Prashant|          9238614990|             15000|             18.5|
+-------+------------------+-------------+--------------------+------------------+-----------------+

